# CRM Data Exploration

Profile the Contacts and Opportunities data from GoHighLevel.

In [ ]:
import pandas as pd
import numpy as np

contacts = pd.read_csv("data/contacts_jan_2026.csv")
opportunities = pd.read_csv("data/Opportunities (1).csv")

# Parse dates (utc=True needed due to mixed timezone offsets)
opportunities["Created on"] = pd.to_datetime(opportunities["Created on"], utc=True)
opportunities["Updated on"] = pd.to_datetime(opportunities["Updated on"], utc=True)
contacts["Created"] = pd.to_datetime(contacts["Created"], utc=True)

print(f"Contacts: {contacts.shape[0]:,} rows, {contacts.shape[1]} columns")
print(f"Opportunities: {opportunities.shape[0]:,} rows, {opportunities.shape[1]} columns")

In [ ]:
# Contacts: null counts and unique values
print("=== Contacts Profile ===")
for col in contacts.columns:
    n = contacts[col].count()
    nunique = contacts[col].nunique()
    pct_null = (1 - n / len(contacts)) * 100
    print(f"{col:20s}  filled={n:>5,}  null={pct_null:5.1f}%  unique={nunique:>5,}")

In [ ]:
# Opportunities: null counts and unique values
print("=== Opportunities Profile ===")
for col in opportunities.columns:
    n = opportunities[col].count()
    nunique = opportunities[col].nunique()
    pct_null = (1 - n / len(opportunities)) * 100
    print(f"{col:40s}  filled={n:>6,}  null={pct_null:5.1f}%  unique={nunique:>6,}")

In [ ]:
# Key categorical distributions
print("=== Status ===")
print(opportunities["status"].value_counts())
print()
print("=== Stage ===")
print(opportunities["stage"].value_counts())
print()
print("=== Lead Type ===")
print(opportunities["Lead Type"].value_counts())
print()
print("=== Project Type ===")
print(opportunities["Project Type"].value_counts())
print()
print("=== Source (top 15) ===")
print(opportunities["source"].value_counts().head(15))

In [ ]:
# Agent assignment distribution
print("=== Assigned Agent ===")
print(opportunities["assigned"].value_counts())
print()
print("=== Pipeline ===")
print(opportunities["pipeline"].value_counts())

In [ ]:
# Date ranges
print("=== Opportunities date range ===")
print(f"Created: {opportunities['Created on'].min()} to {opportunities['Created on'].max()}")
print(f"Updated: {opportunities['Updated on'].min()} to {opportunities['Updated on'].max()}")
print()
print("=== Contacts date range ===")
print(f"Created: {contacts['Created'].min()} to {contacts['Created'].max()}")

In [ ]:
# Contacts per opportunity count (how many contacts have multiple opps?)
opps_per_contact = opportunities.groupby("Contact ID").size()
print("=== Opportunities per Contact ===")
print(opps_per_contact.describe())
print()
print("Distribution:")
print(opps_per_contact.value_counts().sort_index())

In [ ]:
# Lost reasons
lost = opportunities[opportunities["status"] == "lost"]
print(f"Lost opportunities: {len(lost):,}")
print()
print("=== Lost Reason ===")
print(lost["lost reason name"].value_counts().head(20))

In [ ]:
# Win rate by source
source_stats = opportunities.groupby("source").agg(
    total=("status", "count"),
    won=("status", lambda x: (x == "won").sum())
)
source_stats["win_rate"] = source_stats["won"] / source_stats["total"]
source_stats = source_stats.sort_values("total", ascending=False)
print("=== Win Rate by Source (sorted by volume) ===")
print(source_stats.head(15).to_string())

In [ ]:
# Win rate by Lead Type
lt_stats = opportunities.groupby("Lead Type").agg(
    total=("status", "count"),
    won=("status", lambda x: (x == "won").sum())
)
lt_stats["win_rate"] = lt_stats["won"] / lt_stats["total"]
lt_stats = lt_stats.sort_values("total", ascending=False)
print("=== Win Rate by Lead Type ===")
print(lt_stats.to_string())
print()

# Win rate by Project Type
pt_stats = opportunities.groupby("Project Type").agg(
    total=("status", "count"),
    won=("status", lambda x: (x == "won").sum())
)
pt_stats["win_rate"] = pt_stats["won"] / pt_stats["total"]
pt_stats = pt_stats.sort_values("total", ascending=False)
print("=== Win Rate by Project Type ===")
print(pt_stats.to_string())

In [ ]:
# Lead Value distribution
print("=== Lead Value ===")
print(opportunities["Lead Value"].describe())
print()
non_zero_values = opportunities[opportunities["Lead Value"] > 0]["Lead Value"]
print(f"Non-zero Lead Values: {len(non_zero_values):,} ({len(non_zero_values)/len(opportunities)*100:.1f}%)")
if len(non_zero_values) > 0:
    print(non_zero_values.describe())

In [ ]:
# Engagement Score distribution
print("=== Engagement Score ===")
print(opportunities["Engagement Score"].describe())
print()
print("By status:")
print(opportunities.groupby("status")["Engagement Score"].describe())

In [ ]:
# Tags analysis
print("=== Contact Tags (sample) ===")
print(contacts["Tags"].value_counts().head(20))
print()
print("=== Opportunity Tags (sample) ===")
print(opportunities["tags"].value_counts().head(20))